In [1]:
import scanpy as sc
import torch
import numpy as np

In [2]:
adata = sc.read_h5ad("data/whole_dataset.h5ad")

# Basic Info
print(f"Shape: {adata.shape}")
print(f"obs columns: {adata.obs.columns.tolist()}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"Data dtype: {adata.X.dtype}")


Shape: (197358, 540)
obs columns: ['barcode', 'label_id', 'n_genes', 'Area', 'BoundingBoxArea', 'ConvexArea', 'EquivalentDiameter', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'Eccentricity', 'Orientation', 'Center_X', 'Center_Y', 'BoundingBoxMinimum_X', 'BoundingBoxMaximum_X', 'BoundingBoxMinimum_Y', 'BoundingBoxMaximum_Y', 'FormFactor', 'Extent', 'Solidity', 'Compactness', 'EulerNumber', 'MaximumRadius', 'MeanRadius', 'MedianRadius', 'FilledArea', 'SpatialMoment_0_0', 'SpatialMoment_0_1', 'SpatialMoment_0_2', 'SpatialMoment_0_3', 'SpatialMoment_1_0', 'SpatialMoment_1_1', 'SpatialMoment_1_2', 'SpatialMoment_1_3', 'SpatialMoment_2_0', 'SpatialMoment_2_1', 'SpatialMoment_2_2', 'SpatialMoment_2_3', 'CentralMoment_0_0', 'CentralMoment_0_1', 'CentralMoment_0_2', 'CentralMoment_0_3', 'CentralMoment_1_0', 'CentralMoment_1_1', 'CentralMoment_1_2', 'CentralMoment_1_3', 'CentralMoment_2_0', 'CentralMoment_2_1', 'CentralMoment_2_2', 'CentralMoment_2_3', 'NormalizedMoment_0_0', 'Normalized

In [4]:
print(adata)
print("---")
print("X dtype:", adata.X.dtype, "min:", adata.X.min(), "max:", adata.X.max())
print("layers:", list(adata.layers.keys()))
print("---")
morph_cols = [c for c in adata.obs.columns
              if c[0].isupper() and "_x" not in c and "_y" not in c]
print(f"# morphological features: {len(morph_cols)}")
print(morph_cols)
print("---")
print(adata.obs[morph_cols].describe().T.head(15))
print("---")
# Likely batch/sample columns
print([c for c in adata.obs.columns if not c[0].isupper()])

AnnData object with n_obs × n_vars = 197358 × 540
    obs: 'barcode', 'label_id', 'n_genes', 'Area', 'BoundingBoxArea', 'ConvexArea', 'EquivalentDiameter', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'Eccentricity', 'Orientation', 'Center_X', 'Center_Y', 'BoundingBoxMinimum_X', 'BoundingBoxMaximum_X', 'BoundingBoxMinimum_Y', 'BoundingBoxMaximum_Y', 'FormFactor', 'Extent', 'Solidity', 'Compactness', 'EulerNumber', 'MaximumRadius', 'MeanRadius', 'MedianRadius', 'FilledArea', 'SpatialMoment_0_0', 'SpatialMoment_0_1', 'SpatialMoment_0_2', 'SpatialMoment_0_3', 'SpatialMoment_1_0', 'SpatialMoment_1_1', 'SpatialMoment_1_2', 'SpatialMoment_1_3', 'SpatialMoment_2_0', 'SpatialMoment_2_1', 'SpatialMoment_2_2', 'SpatialMoment_2_3', 'CentralMoment_0_0', 'CentralMoment_0_1', 'CentralMoment_0_2', 'CentralMoment_0_3', 'CentralMoment_1_0', 'CentralMoment_1_1', 'CentralMoment_1_2', 'CentralMoment_1_3', 'CentralMoment_2_0', 'CentralMoment_2_1', 'CentralMoment_2_2', 'CentralMoment_2_3', 'Normalized

/Users/hannesneumann/Repositories/conditional-flow-matching-seminar/.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


                         count          mean          std         min  \
Area                  197358.0     68.663971    41.229717    2.000000   
BoundingBoxArea       197358.0    107.340792    69.706181    2.000000   
ConvexArea            197358.0     76.373149    46.320890    2.000000   
EquivalentDiameter    197358.0      8.982754     2.595353    1.595769   
Perimeter             197358.0     30.406660    10.627031    0.000000   
MajorAxisLength       197358.0     11.904040     4.115646    2.000000   
MinorAxisLength       197358.0      7.279627     2.361339    0.000000   
Eccentricity          197358.0      0.733065     0.159670    0.000000   
Orientation           197358.0      1.831994    53.104917  -90.000000   
Center_X              197358.0   5662.714318  2900.940768   83.794872   
Center_Y              197358.0  10303.872914  4646.609686  143.926471   
BoundingBoxMinimum_X  197358.0   5658.136260  2901.024214   81.000000   
BoundingBoxMaximum_X  197358.0   5668.281661  2900.

In [5]:
adata.obsm["X_pca"].shape[1]

50

In [6]:
adata.obs_keys

<bound method AnnData.obs_keys of AnnData object with n_obs × n_vars = 197358 × 540
    obs: 'barcode', 'label_id', 'n_genes', 'Area', 'BoundingBoxArea', 'ConvexArea', 'EquivalentDiameter', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'Eccentricity', 'Orientation', 'Center_X', 'Center_Y', 'BoundingBoxMinimum_X', 'BoundingBoxMaximum_X', 'BoundingBoxMinimum_Y', 'BoundingBoxMaximum_Y', 'FormFactor', 'Extent', 'Solidity', 'Compactness', 'EulerNumber', 'MaximumRadius', 'MeanRadius', 'MedianRadius', 'FilledArea', 'SpatialMoment_0_0', 'SpatialMoment_0_1', 'SpatialMoment_0_2', 'SpatialMoment_0_3', 'SpatialMoment_1_0', 'SpatialMoment_1_1', 'SpatialMoment_1_2', 'SpatialMoment_1_3', 'SpatialMoment_2_0', 'SpatialMoment_2_1', 'SpatialMoment_2_2', 'SpatialMoment_2_3', 'CentralMoment_0_0', 'CentralMoment_0_1', 'CentralMoment_0_2', 'CentralMoment_0_3', 'CentralMoment_1_0', 'CentralMoment_1_1', 'CentralMoment_1_2', 'CentralMoment_1_3', 'CentralMoment_2_0', 'CentralMoment_2_1', 'CentralMoment_2_2'

In [9]:
adata.obs['barcode'].nunique()

197358

In [33]:
_DROP_COLUMNS = {
    "Center_X", "Center_Y",
    "BoundingBoxMinimum_X", "BoundingBoxMaximum_X",
    "BoundingBoxMinimum_Y", "BoundingBoxMaximum_Y",
    "Orientation",
    "NormalizedMoment_1_0", "NormalizedMoment_0_0", "NormalizedMoment_0_1"
}


def get_morphology_columns(obs_columns: list[str]) -> list[str]:
    cols = [c for c in obs_columns if c[0].isupper()]
    cols = [c for c in cols if c not in _DROP_COLUMNS]
    cols = [c for c in cols if not c.startswith("SpatialMoment_")]
    return cols

morph_cols = get_morphology_columns(list(adata.obs.columns))
y = torch.tensor(adata.obs[morph_cols].values.astype(np.float32), dtype=torch.float32)

In [ ]:
torch.isnan(y).sum().item()
c_train, y_train, c_val, y_val, y_mean, y_std, morph_cols = 


print(f"Train: {len(c_train):,}  |  Val: {len(c_val):,}")

0

In [36]:
# Auf dem raw dataframe vor Standardisierung
nan_per_col = adata.obs.isna().sum()
print(nan_per_col[nan_per_col > 0].sort_values(ascending=False))
print(f"\nRows with any NaN: {adata.obs.isna().any(axis=1).sum()} / {len(adata.obs)}")

NormalizedMoment_0_0    197358
NormalizedMoment_0_1    197358
NormalizedMoment_1_0    197358
dtype: int64

Rows with any NaN: 197358 / 197358


In [24]:
adata.obs.isna().sum().sum()

np.int64(592074)

In [29]:
row = adata.obs.iloc[5]
print(row[row.isna()])

NormalizedMoment_0_0    NaN
NormalizedMoment_0_1    NaN
NormalizedMoment_1_0    NaN
Name: 5, dtype: object


In [32]:
adata.obs['NormalizedMoment_1_0']

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
          ..
197353   NaN
197354   NaN
197355   NaN
197356   NaN
197357   NaN
Name: NormalizedMoment_1_0, Length: 197358, dtype: float64

In [4]:
adata.X.shape

(197358, 540)